In [1]:
from dotenv import load_dotenv
import os
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic import hub
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.runnables import RunnablePassthrough 
from langchain_core.output_parsers import StrOutputParser
from pprint import pprint

C:\Users\smkamran\AppData\Local\Temp\ipykernel_24076\3562791569.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader


In [2]:
load_dotenv()  # Load environment variables from .env file
api_key = os.getenv("OPEN_AI_API_KEY")
url_to_scrape = os.getenv("TEST_URL_TO_SCRAPE")
print(api_key[0:5] + "...")
print(url_to_scrape)

sk-pr...
https://books.toscrape.com/


## Scrape a web page

In [3]:
loader = WebBaseLoader(web_paths=[url_to_scrape])
docs = loader.load()

In [4]:
for doc in docs:
    pprint(f"Document Metadata: {doc.metadata}")
    pprint(f"Document Content: {doc.page_content}")
    print("=" * 80)  # Separator between documents

("Document Metadata: {'source': 'https://books.toscrape.com/', 'title': "
 "'\\n    All products | Books to Scrape - Sandbox\\n', 'description': '', "
 "'language': 'en-us'}")
('Document Content: \n'
 '\n'
 '\n'
 '\n'
 '  \n'
 '\n'
 '\n'
 '    All products | Books to Scrape - Sandbox\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 'Books to Scrape We love being scraped!\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 'Home\n'
 '\n'
 'All products\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '                            \n'
 '                                Books\n'
 '                            \n'
 '                        \n'
 '\n'
 '\n'
 '\n'
 '                            \n'
 '                                Travel\n'
 '                            \n'
 '                        \n'
 '\n'
 '\n'
 '\n'
 '                            \n'
 '                                Mystery\n'
 '                            \n

## Split and do Chunking

In [5]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

In [6]:
splits = text_splitter.split_documents(docs)

In [7]:
# print the splits using list comprehension
pprint(len(splits))
[pprint(split) for split in splits]

12
Document(metadata={'source': 'https://books.toscrape.com/', 'title': '\n    All products | Books to Scrape - Sandbox\n', 'description': '', 'language': 'en-us'}, page_content='All products | Books to Scrape - Sandbox\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nBooks to Scrape We love being scraped!\n\n\n\n\n\n\n\n\nHome\n\nAll products\n\n\n\n\n\n\n\n\n\n                            \n                                Books\n                            \n                        \n\n\n\n                            \n                                Travel\n                            \n                        \n\n\n\n                            \n                                Mystery\n                            \n                        \n\n\n\n                            \n                                Historical Fiction\n                            \n                        \n\n\n\n                            \n                                Sequential Art\n                            \n 

[None, None, None, None, None, None, None, None, None, None, None, None]

## Save the chunks to vectorstore using OpenAIEmbeddings

An API key is required

In [ ]:
vectorstore = Chroma.from_documents(
    documents=splits, 
    embedding=OpenAIEmbeddings(api_key=api_key)
)

In [ ]:
print(f"Vectorstore created with {vectorstore.collection.count()} documents.")

In [ ]:
ids = vectorstore._collection.get()
print(ids)

In [ ]:
for i, id in enumerate(ids):
    print(f"Document # {i+1:02d} | ID: {id} | Content: {vectorstore._collection.get(id)}")

## Retrieve Context using vector Search

In [ ]:
# Retrieve an existing prompt from LangSmith
from langsmith import Client

client = Client()

prompt = client.pull_prompt(
    "rlm/rag-prompt", 
    dangerously_pull_public_prompt=True
)

type(prompt)

langchain_core.prompts.chat.ChatPromptTemplate

In [ ]:
pprint(prompt.messages[0].prompt.template)

('You are an assistant for question-answering tasks. Use the following pieces '
 "of retrieved context to answer the question. If you don't know the answer, "
 "just say that you don't know. Use three sentences maximum and keep the "
 'answer concise.\n'
 'Question: {question} \n'
 'Context: {context} \n'
 'Answer:')


In [ ]:
retriever = vectorstore.as_retriever()

In [ ]:
llm = ChatOpenAI(
    model_name="gpt-4o",
    temperature=0.0,
    api_key=api_key
)

In [ ]:
def format_docs(docs):
    return "\n".join(doc.page_content)

## Create Rag Chain and invoke the query

In [ ]:
rag_chain = ({"context": retriever | format_docs, "question": RunnablePassthrough()} 
             | prompt 
             | llm 
             | StrOutputParser())

In [ ]:
rag_chain.invoke({"question": "What is the main topic of the page?"})